# NBA 3.1 — Pipeline Configuration

Single source of truth for all parameters, cohort logic, and helper functions
used across the patient journey (NB01–NB06) and HCP feature engineering notebooks.

**Location:** `/kerendia_nba_3_1_develop_clone/00_cohort_config`  
**Usage:** `%run "../00_cohort_config"` from any subfolder notebook.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DateType
from dateutil.relativedelta import relativedelta
from datetime import date, datetime, timedelta
import calendar
import builtins
import sys

## Section 1 — Global Parameters
Shared across patient journey and HCP feature notebooks.

In [0]:
# ============================================================
# GLOBAL PARAMETERS — change these to configure the pipeline
# ============================================================

OFFSET_DAY       = 7              # Day-of-month offset (0 = standard calendar months)
DATA_START_DATE  = "2023-01-01"   # Exclude all data before this date
COHORT_ORIGIN    = "2025-01-01"   # Reference month of C1
COHORT_LAST      = "2026-10-01"   # Reference month of C22 (last cohort)
COHORT_MONTH     = "2025-02"      # Single HCP cohort (NB01; None = skip HCP window computation)
COHORT_MONTHS    = None           # Multi-cohort runs (NB02+): None = all cohorts COHORT_ORIGIN..COHORT_LAST
                                  #   with complete data; or a list, e.g. ["2025-01", "2025-02"]
VALIDATION_MODE  = True           # True = filter to 10 test HCPs; False = production (all HCPs)

print(f"OFFSET_DAY       = {OFFSET_DAY}")
print(f"DATA_START_DATE  = {DATA_START_DATE}")
print(f"COHORT_ORIGIN    = {COHORT_ORIGIN}")
print(f"COHORT_LAST      = {COHORT_LAST}")
print(f"COHORT_MONTH     = {COHORT_MONTH}")
print(f"COHORT_MONTHS    = {COHORT_MONTHS if COHORT_MONTHS is not None else 'all (complete data only)'}")
print(f"VALIDATION_MODE  = {VALIDATION_MODE}")

## Section 2 — Cohort Functions
`assign_cohort_month()` maps dates to offset-aligned months. Used by both patient journey and HCP features.

In [0]:
def assign_cohort_month(svc_dt_col, offset_days=None):
    """
    Maps a service-date column to its cohort reference month.

    With OFFSET_DAY = 0  → standard trunc to calendar month.
    With OFFSET_DAY = 7  → cohort boundaries fall on the 7th:
        C1 = [Jan 7, Feb 7),  C2 = [Feb 7, Mar 7), …

    The 'offset-th' day belongs to the NEW cohort (start-inclusive).
    offset_days=None uses the global OFFSET_DAY; pass a value (e.g. cb["offset_days"])
    to evaluate another offset, such as the offset-0 parity checks.
    """
    _off = OFFSET_DAY if offset_days is None else offset_days
    if _off == 0:
        return F.trunc(svc_dt_col, "month")
    else:
        return F.trunc(F.date_sub(svc_dt_col, _off - 1), "month")


def _cohort_month_sql(col_name):
    """
    Returns the SQL expression string for assign_cohort_month.
    Use inside F.expr() where a PySpark Column cannot be used (e.g. sequence()).
    """
    if OFFSET_DAY == 0:
        return f"trunc({col_name}, 'month')"
    else:
        return f"trunc(date_sub({col_name}, {OFFSET_DAY - 1}), 'month')"

In [0]:
# Build the cohort lookup: C1 through C22
_origin = date.fromisoformat(COHORT_ORIGIN)
_last   = date.fromisoformat(COHORT_LAST)

cohort_rows = []
_current = _origin
_idx = 1
while _current <= _last:
    _next = _current + relativedelta(months=1)
    if OFFSET_DAY == 0:
        cohort_start = _current
        cohort_end   = _next
    else:
        cohort_start = _current.replace(day=OFFSET_DAY)
        cohort_end   = _next.replace(day=OFFSET_DAY)
    cohort_rows.append((f"C{_idx}", _current, cohort_start, cohort_end))
    _current = _next
    _idx += 1

schema = StructType([
    StructField("cohort_label",      StringType(), False),
    StructField("cohort_ref_month",  DateType(),   False),
    StructField("cohort_start",      DateType(),   False),
    StructField("cohort_end",        DateType(),   False),
])

df_cohort_ref = spark.createDataFrame(cohort_rows, schema=schema)
print(f"{len(cohort_rows)} cohorts defined (C1 – C{len(cohort_rows)})")

## Section 3 — Validation / Production Filters
Controlled by `VALIDATION_MODE` in Section 1.  
* `get_test_npis()` — returns the 10 NPI list or `None` (data source)  
* `apply_hcp_filter(df)` — patient-level filter (for patient journey notebooks: finds patients linked to test NPIs)  
* `apply_npi_filter(df)` — NPI-level filter (for HCP feature notebooks: filters directly on NPI column)  

Both filter functions no-op in production mode.

In [0]:
# ============================================================
# VALIDATION / PRODUCTION FILTERS
# Controlled by VALIDATION_MODE in Section 1.
# ============================================================

def get_test_npis():
    """Return the 10 validation HCPs when in validation mode, None for production."""
    if not VALIDATION_MODE:
        return None
    return [
        "1396374062", "1508230566", "1801062104", "1386651560", "1528629052",
        "1659887388", "1215675616", "1467950774", "1235697863", "1841364841"
    ]

# Expose TEST_NPIS for backward compatibility (None in production mode)
TEST_NPIS = get_test_npis()


# --- Patient-level filter (patient journey notebooks) ---
def apply_hcp_filter(df, npi_col="NPI_NUMBER", patient_col="PATIENT_ID"):
    """
    Filters a DataFrame to only patients linked to the test HCPs.
    Use in patient journey notebooks where data is at patient grain.
    In production mode (VALIDATION_MODE=False), returns the unfiltered df.
    Returns (filtered_df, patient_list).
    """
    npis = get_test_npis()
    if npis is None:
        print("Production mode — no patient filter applied")
        return df, None
    test_patients = [
        row[patient_col] for row in
        df.filter(F.col(npi_col).isin(npis))
          .select(patient_col).distinct().collect()
    ]
    df_filtered = df.filter(F.col(patient_col).isin(test_patients))
    print(f"Validation mode — filtered to {len(test_patients)} patients across {len(npis)} HCPs")
    return df_filtered, test_patients


# --- NPI-level filter (HCP feature notebooks) ---
def apply_npi_filter(df, npi_col="NPI_NUMBER"):
    """
    Filters a DataFrame directly on NPI_NUMBER.
    Use in HCP feature notebooks where data is at NPI grain.
    In production mode (VALIDATION_MODE=False), returns the unfiltered df.
    """
    npis = get_test_npis()
    if npis is None:
        print("Production mode — no NPI filter applied")
        return df
    df_filtered = df.filter(F.col(npi_col).isin(npis))
    print(f"Validation mode — filtered to {len(npis)} test NPIs")
    return df_filtered

## Section 4 — HCP Feature Engineering Helpers
Reusable functions for the claims monthly notebooks (01–06).

In [0]:
# ============================================================
# HCP FEATURE HELPERS — Reusable across all claims notebooks
# ============================================================

def clean_col_name(name):
    """Apply the same column-name cleaning as the original notebooks.
    Parentheses removal comes from 02_claim_lab_monthly (e.g. 'STAGE 2 (MILD)');
    no Rx values contain parentheses, so NB01 names are unchanged."""
    return (name
            .replace(", ", " ")
            .replace("(", "")
            .replace(")", "")
            .replace("+", "and")
            .replace("/", " or")
            .replace("-", "_")
            .replace(" ", "_")
            .replace("__", "_")
            .upper())


def pivoted_counts(df, npi_col, pivot_col, prefix, suffix, values=None):
    """
    Pivot df on pivot_col, count claims per NPI, clean column names.
    values=None: collect the distinct pivot values present in df (NB01 behaviour).
    values=list: use that fixed list, so every cohort/window gets the same columns
                 (values with no claims become 0).
    """
    if values is None:
        _vals = sorted(
            [r[0] for r in df.select(pivot_col).distinct().collect() if r[0] is not None]
        )
    else:
        _vals = sorted(values)
    df_piv = (
        df.groupBy(npi_col)
        .pivot(pivot_col, values=_vals)
        .agg(F.count(F.lit(1)).alias("cnt"))
        .na.fill(0)
    )
    new_names = [npi_col] + [
        f"{prefix}{clean_col_name(v)}{suffix}" for v in _vals
    ]
    return df_piv.toDF(*new_names)


def filter_to_window(df, svc_dt_col, w_start, w_end):
    """Filter DataFrame to a date-range window (exclusive start, inclusive end)."""
    return df.filter(
        (F.col(svc_dt_col) > F.lit(w_start)) &
        (F.col(svc_dt_col) <= F.lit(w_end))
    )


def build_feature_group(df_rx, npi_col, svc_dt_col, pivot_col, prefix,
                        window_bounds, extra_filter=None, combine_cols=None,
                        pivot_values=None):
    """
    Build one feature group across all lookback windows.
    Filters claims to each date range, optionally applies extra_filter,
    optionally creates a compound pivot column, then pivots and counts.
    """
    window_dfs = []
    for w in sorted(window_bounds.keys()):
        bounds = window_bounds[w]
        df_w = filter_to_window(df_rx, svc_dt_col, bounds["start"], bounds["end"])
        if extra_filter:
            df_w = df_w.filter(extra_filter)
        actual_pivot = pivot_col
        if combine_cols:
            df_w = df_w.withColumn(
                "_pivot_combined",
                F.concat_ws("_", *[F.col(c) for c in combine_cols])
            )
            actual_pivot = "_pivot_combined"
        suffix = f"_IN_LAST_{w}_MONTH"
        df_features = pivoted_counts(df_w, npi_col, actual_pivot, prefix, suffix,
                                     values=pivot_values)
        window_dfs.append(df_features)
    combined = window_dfs[0]
    for df_next in window_dfs[1:]:
        combined = combined.join(df_next, on=npi_col, how="outer")
    return combined.na.fill(0)


def build_total_col(df_rx, npi_col, svc_dt_col, window_bounds, total_name="TOTAL_RX_CLAIMS"):
    """Build {total_name}_IN_LAST_{N}_MONTH per NPI per window (denominators for PCT).
    total_name defaults to TOTAL_RX_CLAIMS (NB01); NB02 uses TOTAL_LAB_CLAIMS."""
    dfs = []
    for w in sorted(window_bounds.keys()):
        bounds = window_bounds[w]
        df_w = filter_to_window(df_rx, svc_dt_col, bounds["start"], bounds["end"])
        df_total = (
            df_w.groupBy(npi_col)
            .agg(F.count(F.lit(1)).alias(f"{total_name}_IN_LAST_{w}_MONTH"))
        )
        dfs.append(df_total)
    combined = dfs[0]
    for df_next in dfs[1:]:
        combined = combined.join(df_next, on=npi_col, how="outer")
    return combined.na.fill(0)

## Section 5 — HCP Cohort Parameters
Lookback windows, cutoff date, and window bounds used by all HCP feature notebooks (01–06).  
Set `COHORT_MONTH` in Section 1. When `None`, this section is skipped (patient journey runs).

In [0]:
# ---- Alias for backward compat ----
OFFSET_DAYS = OFFSET_DAY

# ---- Column-name constants ----
NPI_COL = "NPI_NUMBER"
SVC_COL = "SVC_DT"

# ---- Lookback config ----
LOOKBACK_WINDOWS = [1, 3, 6, 9, 12]
AVG_EXTRA_MONTH  = True   # True = replicate old step3_lag_avg (1 month wider)

# ---- Window bounds (only when COHORT_MONTH is set) ----
def _build_bounds(lookback_windows, cutoff, cohort_start, extra_month=False, offset_days=None):
    """
    Build {window_size: {"start": date, "end": date}} for each lookback window.
    At offset 0 the exclusive start snaps to calendar-month boundaries.
    At offset > 0 the start is cutoff minus N calendar months.
    When extra_month is True (AVG behaviour), the window extends one month earlier.
    offset_days=None uses the global OFFSET_DAYS.
    """
    _off = OFFSET_DAYS if offset_days is None else offset_days
    bounds = {}
    for w in lookback_windows:
        end = cutoff
        if _off == 0:
            months_back = w - 1
            if extra_month:
                months_back += 1
            start_month = cohort_start - relativedelta(months=months_back)
            start = start_month - timedelta(days=1)
        else:
            months_back = w
            if extra_month:
                months_back += 1
            start = cutoff - relativedelta(months=months_back)
        bounds[w] = {
            "start": start.strftime("%Y-%m-%d"),
            "end":   end.strftime("%Y-%m-%d"),
        }
    return bounds

def _cohort_cutoff(cohort_start, off):
    """
    Last day (inclusive) of a cohort, consistent with assign_cohort_month():
    the offset-th day starts the NEW cohort.
      offset 0 -> month end                  (Feb cohort: up to Feb 28)
      offset 7 -> month end + (offset - 1)   (Feb cohort: up to Mar 6; Mar 7 starts March)
    """
    _month_end = datetime(cohort_start.year, cohort_start.month,
                          calendar.monthrange(cohort_start.year, cohort_start.month)[1])
    return _month_end + timedelta(days=(off - 1) if off > 0 else 0)


def get_cohort_bounds(cohort_month, offset_days=None):
    """
    All dates for one cohort month ('YYYY-MM'):
    cohort_id, cutoff (last day of the cohort, see _cohort_cutoff), window_bounds,
    avg_window_bounds. Same rules as the single-cohort globals below.
    """
    _off = OFFSET_DAYS if offset_days is None else offset_days
    _cs  = datetime.strptime(f"{cohort_month}-01", "%Y-%m-%d")
    _cut = _cohort_cutoff(_cs, _off)
    return {
        "cohort_month":      cohort_month,
        "cohort_id":         f"{cohort_month}_offset{_off}",
        "offset_days":       _off,
        "cutoff":            _cut,
        "window_bounds":     _build_bounds(LOOKBACK_WINDOWS, _cut, _cs, False, _off),
        "avg_window_bounds": _build_bounds(LOOKBACK_WINDOWS, _cut, _cs, AVG_EXTRA_MONTH, _off),
    }


def get_cohort_months(max_data_date=None, offset_days=None):
    """
    Cohort months to process: COHORT_MONTHS if set, else every month from
    COHORT_ORIGIN to COHORT_LAST. If max_data_date is given, cohorts whose cutoff
    is after it (incomplete most-recent window) are dropped.
    """
    if COHORT_MONTHS is not None:
        months = list(COHORT_MONTHS)
    else:
        months, _m = [], date.fromisoformat(COHORT_ORIGIN)
        while _m <= date.fromisoformat(COHORT_LAST):
            months.append(_m.strftime("%Y-%m"))
            _m += relativedelta(months=1)
    if max_data_date is not None:
        _max = datetime.combine(max_data_date, datetime.min.time()) if isinstance(max_data_date, date) and not isinstance(max_data_date, datetime) else max_data_date
        months = [m for m in months if get_cohort_bounds(m, offset_days)["cutoff"] <= _max]
    return months


if COHORT_MONTH is not None:
    _cohort_start = datetime.strptime(f"{COHORT_MONTH}-01", "%Y-%m-%d")
    CUTOFF_DATE   = _cohort_cutoff(_cohort_start, OFFSET_DAYS)
    COHORT_ID     = f"{COHORT_MONTH}_offset{OFFSET_DAYS}"

    WINDOW_BOUNDS     = _build_bounds(LOOKBACK_WINDOWS, CUTOFF_DATE, _cohort_start, extra_month=False)
    AVG_WINDOW_BOUNDS = _build_bounds(LOOKBACK_WINDOWS, CUTOFF_DATE, _cohort_start, extra_month=AVG_EXTRA_MONTH)

    print(f"Cohort: {COHORT_MONTH} | Offset: {OFFSET_DAYS} days | "
          f"Cutoff: {CUTOFF_DATE.strftime('%Y-%m-%d')} | ID: {COHORT_ID}")
    for w in LOOKBACK_WINDOWS:
        wb = WINDOW_BOUNDS[w]
        ab = AVG_WINDOW_BOUNDS[w]
        avg_months = w + (1 if AVG_EXTRA_MONTH else 0)
        print(f"  Last {w:2d}m: {wb['start']} -> {wb['end']}  "
              f"(AVG: {ab['start']} -> {ab['end']}, /{avg_months + 1})")
else:
    CUTOFF_DATE       = None
    COHORT_ID         = None
    WINDOW_BOUNDS     = None
    AVG_WINDOW_BOUNDS = None
    print("COHORT_MONTH is None — HCP window bounds skipped (patient journey mode)")

In [0]:
# Quick self-test: verify assign_cohort_month with known examples
from datetime import date as _d

_test_cases = [
    ("2025-01-01", 0,  "2025-01-01"),
    ("2025-01-07", 7,  "2025-01-01"),
    ("2025-01-06", 7,  "2024-12-01"),
    ("2025-02-06", 7,  "2025-01-01"),
    ("2025-02-07", 7,  "2025-02-01"),
    ("2025-10-04", 7,  "2025-09-01"),
]

print(f"Testing with current OFFSET_DAY = {OFFSET_DAY}")
print("-" * 55)
for svc, off, expected in _test_cases:
    if off != OFFSET_DAY:
        continue
    test_df = spark.createDataFrame([(svc,)], ["SVC_DT"]).withColumn("SVC_DT", F.to_date("SVC_DT"))
    result = test_df.select(assign_cohort_month(col("SVC_DT")).alias("cohort_month")).collect()[0][0]
    status = "PASS" if str(result) == expected else "FAIL"
    print(f"  {svc} → {result}  (expected {expected})  [{status}]")

# HCP window boundary must agree with assign_cohort_month(): every day in a cohort's
# 1-month window maps to that cohort, and the days just outside map to its neighbours.
# Pure Python (same rule as assign_cohort_month), checked for every cohort.
def _py_cohort_month(d, off):
    return (d - timedelta(days=off - 1)).replace(day=1) if off > 0 else d.replace(day=1)

_bad = []
for _m in get_cohort_months():
    _cb  = get_cohort_bounds(_m)
    _s   = datetime.strptime(_cb["window_bounds"][1]["start"], "%Y-%m-%d")   # exclusive
    _e   = datetime.strptime(_cb["window_bounds"][1]["end"], "%Y-%m-%d")     # inclusive
    _ref = datetime.strptime(f"{_m}-01", "%Y-%m-%d")
    if not (_py_cohort_month(_s + timedelta(days=1), OFFSET_DAY) == _ref    # first day in
            and _py_cohort_month(_e, OFFSET_DAY) == _ref                    # last day in
            and _py_cohort_month(_s, OFFSET_DAY) != _ref                    # day before out
            and _py_cohort_month(_e + timedelta(days=1), OFFSET_DAY) != _ref):  # day after out
        _bad.append(_m)
_ex = get_cohort_bounds("2025-02")
print(f"\nHCP window vs cohort assignment ({len(get_cohort_months())} cohorts): "
      f"{'PASS' if not _bad else 'FAIL ' + str(_bad)}")
print(f"  e.g. 2025-02: 1-month window ({_ex['window_bounds'][1]['start']}, {_ex['window_bounds'][1]['end']}], "
      f"cutoff {_ex['cutoff']:%Y-%m-%d}")

print("\n✅ Config loaded. Available to all pipeline notebooks.")